# Q15. Snowpark UDF - Inférence dans Snowflake

Crée une UDF Python qui :
1. Prend un chemin de Stage en entrée (`@RESPIRATORY_STAGE/asthma/file.wav`)
2. Lit le fichier WAV depuis le Stage
3. Applique le même preprocessing qu'à l'entraînement
4. Charge le modèle ConvNeXt depuis le Stage
5. Retourne les probabilités pour les 5 classes

In [1]:
import os
import getpass
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark.types import StringType, VariantType
from snowflake.snowpark.functions import udf

load_dotenv('../.env')
print('Librairies chargées.')

Librairies chargées.


In [2]:
totp = getpass.getpass('Code MFA (6 chiffres) : ')

connection_params = {
    'account'      : os.environ['SNOWFLAKE_ACCOUNT'],
    'user'         : os.environ['SNOWFLAKE_USER'],
    'password'     : os.environ['SNOWFLAKE_PASSWORD'],
    'authenticator': os.environ['SNOWFLAKE_AUTHENTICATOR'],
    'passcode'     : totp,
    'warehouse'    : os.environ['SNOWFLAKE_WAREHOUSE'],
    'database'     : os.environ['SNOWFLAKE_DATABASE'],
    'schema'       : os.environ['SNOWFLAKE_SCHEMA'],
    'role'         : os.environ['SNOWFLAKE_ROLE'],
}

session = Session.builder.configs(connection_params).create()
print(f'Connecté : {session.get_current_database()}.{session.get_current_schema()}')

 pip install snowflake-connector-python[secure-local-storage]


Connecté : "M2_UQAC_EQUIPE_1_DB"."PUBLIC"


## Vérification des packages disponibles dans Snowflake Anaconda

In [3]:
packages_to_check = ['torch', 'torchvision', 'torchaudio', 'librosa', 'scipy', 'numpy', 'snowflake-snowpark-python']
result = session.sql(
    f"SELECT * FROM information_schema.packages WHERE package_name IN ({','.join(repr(p) for p in packages_to_check)})"
).to_pandas()
print(result[['PACKAGE_NAME', 'VERSION']].drop_duplicates().to_string(index=False))


             PACKAGE_NAME  VERSION
                    numpy   1.16.6
                    numpy   1.19.2
                    numpy   1.19.5
                    numpy   1.20.1
                    numpy   1.20.2
                    numpy   1.20.3
                    numpy   1.21.2
                    numpy   1.21.5
                    numpy   1.21.6
                    numpy   1.22.3
                    numpy   1.23.1
                    numpy   1.23.3
                    numpy   1.23.4
                    numpy   1.23.5
                    numpy   1.24.3
                    numpy   1.25.0
                    numpy   1.25.2
                    numpy   1.26.0
                    numpy   1.26.2
                    numpy   1.26.3
                    numpy   1.26.4
                    numpy    2.0.0
                    numpy    2.0.1
                    numpy    2.0.2
                    numpy    2.1.1
                    numpy    2.1.3
                    numpy    2.2.1
                    

## Création de l'UDF Snowpark

L'UDF utilise `SnowflakeFile` pour lire le WAV depuis le Stage et `librosa` pour le preprocessing (torchaudio n'est pas requis).

In [ ]:
# Vérifie si les paquets "premium" d'Anaconda sont visibles
session.sql("""
    SELECT * FROM information_schema.packages 
    WHERE package_name IN ('torch', 'librosa') 
    AND language = 'python'
""").show()

In [ ]:
STAGE_NAME = 'RESPIRATORY_STAGE'

session.clear_packages()
session.clear_imports()

# torchvision (inclut torch) + scipy + numpy — natifs Snowflake Python 3.13
# torchaudio exclu : incompatible Python 3.13
session.add_packages('torchvision', 'scipy', 'numpy', 'snowflake-snowpark-python')

@udf(
    name='PREDICT_RESPIRATORY',
    is_permanent=True,
    stage_location=f'@{STAGE_NAME}/udf/',
    imports=[f'@{STAGE_NAME}/models/best_model_convnext.pth'],
    input_types=[StringType()],
    return_type=VariantType(),
    replace=True,
    session=session,
)
def predict_respiratory(stage_path: str) -> dict:
    import sys, io, os
    from math import gcd
    import numpy as np
    import torch
    import torch.nn as nn
    from torchvision.models import convnext_tiny
    from scipy.signal import butter, filtfilt, resample_poly
    from scipy.io import wavfile
    from snowflake.snowpark.files import SnowflakeFile

    class RespiratoryModel(nn.Module):
        def __init__(self, num_classes=5):
            super().__init__()
            self.cnn = convnext_tiny(weights=None)
            old_conv = self.cnn.features[0][0]
            self.cnn.features[0][0] = nn.Conv2d(1, old_conv.out_channels, kernel_size=4, stride=4)
            self.cnn.classifier = nn.Identity()
            self.classifier = nn.Sequential(
                nn.LayerNorm((768, 1, 1), eps=1e-6),
                nn.Flatten(1),
                nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(256, 64),  nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(64, num_classes)
            )
        def forward(self, x):
            return self.classifier(self.cnn(x))

    SR         = 22050
    TARGET_LEN = SR * 6
    CLASSES    = ['asthma', 'bronchial', 'copd', 'healthy', 'pneumonia']

    def load_audio(audio_bytes, target_sr=SR):
        sr, data = wavfile.read(io.BytesIO(audio_bytes))
        if data.ndim > 1:
            data = data.mean(axis=1)
        max_val = float(np.iinfo(data.dtype).max) if np.issubdtype(data.dtype, np.integer) else 1.0
        data = data.astype(np.float32) / max_val
        if sr != target_sr:
            g = gcd(int(sr), int(target_sr))
            data = resample_poly(data, target_sr // g, sr // g).astype(np.float32)
        return data

    def trim_silence(audio, top_db=20, hop_length=512, frame_length=2048):
        energies = np.array([
            np.mean(audio[i:i+frame_length]**2)
            for i in range(0, max(1, len(audio) - frame_length + 1), hop_length)
        ])
        threshold = np.max(energies) * 10**(-top_db / 10)
        mask = energies > threshold
        if not mask.any():
            return audio
        start = np.argmax(mask) * hop_length
        end   = (len(mask) - np.argmax(mask[::-1])) * hop_length
        return audio[start:min(end, len(audio))]

    def bandpass_filter(audio, lowcut=100, highcut=2000, sr=SR, order=4):
        nyq = sr / 2.0
        b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
        return filtfilt(b, a, audio).astype(np.float32)

    def pad_or_crop(audio, target_len=TARGET_LEN):
        if len(audio) < target_len:
            pad = target_len - len(audio)
            audio = np.pad(audio, (pad // 2, pad - pad // 2))
        else:
            audio = audio[:target_len]
        return audio

    def compute_mel(audio, sr=SR, n_fft=1024, hop_length=512, n_mels=128, top_db=80.0):
        """Réplique torchaudio.MelSpectrogram + AmplitudeToDB(top_db=80)."""
        audio_pad = np.pad(audio, n_fft // 2, mode='reflect')
        window    = np.hanning(n_fft)
        n_frames  = 1 + len(audio) // hop_length
        power     = np.zeros((n_fft // 2 + 1, n_frames), dtype=np.float32)
        for i in range(n_frames):
            s = i * hop_length
            f = audio_pad[s:s+n_fft]
            if len(f) < n_fft:
                f = np.pad(f, (0, n_fft - len(f)))
            power[:, i] = np.abs(np.fft.rfft(f * window))**2

        hz_to_mel = lambda f: 2595.0 * np.log10(1.0 + f / 700.0)
        mel_to_hz = lambda m: 700.0 * (10.0**(m / 2595.0) - 1.0)
        mel_pts   = np.linspace(hz_to_mel(0), hz_to_mel(sr / 2), n_mels + 2)
        hz_pts    = mel_to_hz(mel_pts)
        bins      = np.floor((n_fft + 1) * hz_pts / sr).astype(int)
        n_freqs   = n_fft // 2 + 1
        fb        = np.zeros((n_mels, n_freqs), dtype=np.float32)
        for m in range(n_mels):
            lo, c, hi = bins[m], bins[m+1], bins[m+2]
            if c > lo:
                fb[m, lo:c] = (np.arange(lo, c) - lo) / (c - lo)
            if hi > c:
                fb[m, c:hi] = (hi - np.arange(c, hi)) / (hi - c)

        mel_spec = fb @ power
        mel_db   = 10.0 * np.log10(np.maximum(mel_spec, 1e-10))
        mel_db   = np.maximum(mel_db, mel_db.max() - top_db)
        return mel_db.astype(np.float32)

    import_dir = sys._xoptions.get('snowflake_import_directory')
    model_path = os.path.join(import_dir, 'best_model_convnext.pth')
    model = RespiratoryModel(num_classes=5)
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.eval()

    with SnowflakeFile.open(stage_path, 'rb') as f:
        audio_bytes = f.read()

    audio = load_audio(audio_bytes)
    audio = trim_silence(audio)
    audio = bandpass_filter(audio)
    audio = audio / (np.max(np.abs(audio)) + 1e-8)
    audio = pad_or_crop(audio)
    mel   = compute_mel(audio)

    tensor = torch.tensor(mel).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.nn.functional.softmax(logits, dim=1).numpy()[0]

    return dict(zip(CLASSES, [float(p) for p in probs]))

print('UDF PREDICT_RESPIRATORY créée avec succès.')


## Test de l'UDF sur un fichier du Stage

In [ ]:
# Test sur un fichier asthme
result = session.sql("""
    SELECT
        stage_path,
        label,
        PREDICT_RESPIRATORY(BUILD_SCOPED_FILE_URL(@RESPIRATORY_STAGE, SUBSTR(stage_path, 20))) AS probabilities
    FROM AUDIO_METADATA
    WHERE label = 'asthma'
    LIMIT 3
""").to_pandas()

print(result.to_string(index=False))

In [ ]:
# Test simple sur un fichier direct
test_path = session.sql("SELECT STAGE_PATH FROM AUDIO_METADATA WHERE LABEL='asthma' LIMIT 1").collect()[0][0]
print(f'Fichier test : {test_path}')

# Construit l'URL scopée pour le fichier
scoped_url = session.sql(f"""
    SELECT BUILD_SCOPED_FILE_URL(@RESPIRATORY_STAGE, '{test_path.replace('@RESPIRATORY_STAGE/', '')}')
""").collect()[0][0]

result = session.sql(f"SELECT PREDICT_RESPIRATORY('{scoped_url}') AS PROBABILITIES").collect()[0][0]
import json
probs = json.loads(result)
classe_predite = max(probs, key=probs.get)

print(f'\nClasse prédite : {classe_predite} ({probs[classe_predite]:.1%})')
for cls, p in sorted(probs.items(), key=lambda x: -x[1]):
    print(f'  {cls:12s}: {p:.1%}')

## Peuplement de la table PREDICTIONS (données simulées)

Insère ~800 prédictions avec des timestamps répartis sur les 30 derniers jours,
en utilisant les vraies pharmacies du CSV pour avoir des coordonnées GPS réelles.

In [4]:
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta

# ── Créer la table si elle n'existe pas ──────────────────────────────────────
session.sql("""
    CREATE TABLE IF NOT EXISTS PREDICTIONS (
        PREDICTION_ID  VARCHAR,
        TIMESTAMP      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        PHARMACIE_ID   VARCHAR,
        CLASSE_PREDITE VARCHAR,
        PROBABILITES   VARIANT,
        CONFIANCE      FLOAT
    )
""").collect()

# ── Vider la table avant de repeupler ────────────────────────────────────────
session.sql("TRUNCATE TABLE PREDICTIONS").collect()
count = session.sql("SELECT COUNT(*) AS N FROM PREDICTIONS").collect()[0][0]
print(f"Table vidée — {count} ligne(s) restante(s).")

# ── Charger les pharmacies ───────────────────────────────────────────────────
pharmacies_df = pd.read_csv('../tessan_pharmacies.csv', encoding='utf-8-sig')
print(f"{len(pharmacies_df)} pharmacies chargées.")

Table vidée — 0 ligne(s) restante(s).
1662 pharmacies chargées.


In [5]:
# ── Générer ~20 prédictions par pharmacie (confiance ≥ 95%) ─────────────────
import uuid

CLASSES = ['asthma', 'bronchial', 'copd', 'healthy', 'pneumonia']
PROBS   = [0.22, 0.10, 0.30, 0.20, 0.18]   # distribution épidémio
np.random.seed(42)

# Exactement 20 prédictions par pharmacie
sample = pharmacies_df.loc[pharmacies_df.index.repeat(20)].reset_index(drop=True)
N      = len(sample)

classes = np.random.choice(CLASSES, size=N, p=PROBS)
# Timestamps répartis aléatoirement sur 30 jours, avec plus de récentes
dates = [
    datetime.now() - timedelta(days=float(np.random.exponential(scale=10)))
    for _ in range(N)
]
# Clamp à 30 jours max
dates = [min(d, datetime.now()) for d in dates]
dates = [max(d, datetime.now() - timedelta(days=30)) for d in dates]

rows = []
for i in range(N):
    # Confiance entre 0.95 et 0.9999 (comme les vraies données)
    confiance = float(np.random.uniform(0.95, 0.9999))
    idx       = CLASSES.index(classes[i])

    # Distribuer le reste (1 - confiance) sur les autres classes
    remainder = 1.0 - confiance
    other_probs = np.random.dirichlet([1] * 4).astype(float) * remainder
    probs = np.zeros(5)
    probs[idx] = confiance
    j = 0
    for k in range(5):
        if k != idx:
            probs[k] = other_probs[j]
            j += 1

    rows.append((
        str(uuid.uuid4()),
        dates[i].strftime('%Y-%m-%d %H:%M:%S'),
        str(sample['PHARMACIE_ID'][i]),
        classes[i],
        json.dumps(dict(zip(CLASSES, probs.tolist()))),
        confiance,
    ))

print(f"{len(rows)} lignes — {len(pharmacies_df)} pharmacies × 20 prédictions")
print(f"Confiance min : {min(r[5] for r in rows):.4f} | max : {max(r[5] for r in rows):.4f}")

33240 lignes — 1662 pharmacies × 20 prédictions
Confiance min : 0.9500 | max : 0.9999


In [9]:
# ── Insertion via table staging + write_pandas ────────────────────────────────
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas

# 1. Construire un DataFrame avec PROBABILITES en VARCHAR
df_insert = pd.DataFrame(rows, columns=[
    'PREDICTION_ID', 'TIMESTAMP', 'PHARMACIE_ID',
    'CLASSE_PREDITE', 'PROBABILITES_STR', 'CONFIANCE'
])
# S'assurer que tous les types sont natifs Python
df_insert['PREDICTION_ID']    = df_insert['PREDICTION_ID'].astype(str)
df_insert['TIMESTAMP']        = df_insert['TIMESTAMP'].astype(str)
df_insert['PHARMACIE_ID']     = df_insert['PHARMACIE_ID'].astype(str)
df_insert['CLASSE_PREDITE']   = df_insert['CLASSE_PREDITE'].astype(str)
df_insert['PROBABILITES_STR'] = df_insert['PROBABILITES_STR'].astype(str)
df_insert['CONFIANCE']        = df_insert['CONFIANCE'].astype(float)

# 2. Écrire dans une table temporaire
conn = session._conn._conn
write_pandas(
    conn, df_insert, 'PREDICTIONS_STAGING',
    auto_create_table=True, overwrite=True,
    quote_identifiers=False,
)
print(f"Table staging créée — {len(df_insert)} lignes.")

# 3. Insérer dans PREDICTIONS avec PARSE_JSON
session.sql("""
    INSERT INTO PREDICTIONS (PREDICTION_ID, TIMESTAMP, PHARMACIE_ID, CLASSE_PREDITE, PROBABILITES, CONFIANCE)
    SELECT
        PREDICTION_ID,
        TO_TIMESTAMP_NTZ(TIMESTAMP),
        PHARMACIE_ID,
        CLASSE_PREDITE,
        PARSE_JSON(PROBABILITES_STR),
        CONFIANCE
    FROM PREDICTIONS_STAGING
""").collect()

# 4. Nettoyer la table temporaire
session.sql("DROP TABLE IF EXISTS PREDICTIONS_STAGING").collect()

count = session.sql("SELECT COUNT(*) AS N FROM PREDICTIONS").collect()[0][0]
print(f"✅ Insertion terminée — {count} lignes dans PREDICTIONS.")

Table staging créée — 33240 lignes.
✅ Insertion terminée — 33240 lignes dans PREDICTIONS.


In [ ]:
session.close()
print('Session fermée.')